# core
> Sync and async HTTP transports over httpx2, and base classes for small REST clients


In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
from fastcore.meta import delegates
from fastcore.basics import store_attr, patch
import asyncio, time, httpx2, json as jsonlib
from functools import partial
from contextlib import nullcontext
from httpx2 import EventSource

The transport raises `ProtocolError` when an SSE payload is not the JSON object shape the protocol promises.


In [ ]:
class ProtocolError(Exception):
    "Raised when a stream payload does not match the expected protocol shape."

In [ ]:
#| hide
import os
from fastcore.test import *
from cachy.core import enable_cachy, doms

In [ ]:
enable_cachy(doms=doms+('jsonplaceholder.typicode.com',))

### `AsyncTransport`

The shared request path for this package and its consumers: one place for header merging, content decoding, multipart behaviour, and SSE streaming, with no provider logic. `request` executes one call and decodes the response by content type (JSON, text, or bytes; `raw=True` returns the `httpx2.Response`). `stream` is an async generator over SSE events: it yields each parsed JSON object until the stream ends, and raises `ProtocolError` on malformed data. The one special case is OpenAI's literal `[DONE]` terminator, a bare non-JSON data line that ends iteration rather than raising.

In [ ]:
class AsyncTransport:
    "Thin async transport over httpx2. By default each request gets a fresh client, so nothing is tied to an event loop; pass `client=` to use a persistent client that you manage (and close via `aclose`)."
    def __init__(self, *, timeout=60.0, client=None, base_headers=None, follow_redirects=True, verify=True, retries=2):
        store_attr('timeout,client,follow_redirects,verify,retries', base_headers=base_headers or {})

    def _client(self):
        if self.client: return nullcontext(self.client)
        return httpx2.AsyncClient(timeout=self.timeout, follow_redirects=self.follow_redirects, verify=self.verify)

    async def aclose(self):
        if self.client: await self.client.aclose()

    def _headers(self, headers=None): return {**self.base_headers, **(headers or {})}

    def _request_headers(self, headers=None, *, files=None):
        "Merge headers and drop explicit content-type for multipart uploads."
        h = self._headers(headers)
        if files is not None:
            # Let httpx compute multipart/form-data with boundary.
            for k in list(h):
                if k.lower() == "content-type": h.pop(k, None)
        return h

    @staticmethod
    def _decode(resp):
        "Decode response body using content type."
        if not resp.content: return None
        ctype = (resp.headers.get("content-type") or "").lower()
        if "application/json" in ctype or ctype.endswith("+json"): return resp.json()
        if ctype.startswith("text/") or "application/x-ndjson" in ctype: return resp.text
        return resp.content

    async def stream(self, method, url, *, headers=None, params=None, json=None, data=None, files=None):
        async with self._client() as client:
            async with client.stream(method, url, headers=self._request_headers(headers, files=files),
                params=params, json=json, data=data, files=files) as resp:
                try: resp.raise_for_status()
                except httpx2.HTTPStatusError as e:
                    try: await resp.aread()
                    except Exception: pass
                    e.args = (f"{e}\n{resp.text}",)
                    raise
                if 'content-type' not in resp.headers: resp.headers['content-type'] = 'text/event-stream'  # some backends (codex) omit it; the old parser never checked
                events = aiter(EventSource(resp, max_event_size=None))
                try:
                    async for event in events:
                        if not event.data: continue
                        if event.data == "[DONE]": return
                        try: raw = jsonlib.loads(event.data)
                        except jsonlib.JSONDecodeError as e: raise ProtocolError(f"Invalid SSE JSON: {e}") from e
                        if isinstance(raw, dict): yield raw
                        else: raise ProtocolError(f"Expected SSE JSON object, got {type(raw).__name__}")
                finally: await events.aclose()

`_send` is one attempt: open a client, send, hand back the response. `_attempt` repeats it when the connection itself fails, up to `retries` times with a short backoff, since nothing has reached the server in that case.


In [ ]:
#| export
@patch
async def _send(self:AsyncTransport, method, url, **kw):
    "One attempt: open a client and send the request"
    async with self._client() as client: return await client.request(method, url, **kw)

@patch
async def _attempt(self:AsyncTransport, method, url, **kw):
    "Send, retrying `retries` times with a backoff when the connection itself fails"
    for i in range(self.retries):
        try: return await self._send(method, url, **kw)
        except (httpx2.ConnectTimeout, httpx2.ConnectError): await asyncio.sleep(0.5 * 2**i)
    return await self._send(method, url, **kw)

`request` is an attempt followed by the status check and decoding. A response that arrives with an error status is the server's answer and is not retried, and neither is a failure after the connection was made, which may have had an effect.

In [ ]:
#| export
@patch
def _result(self:AsyncTransport, resp, raw):
    "Raise on an error status, else the decoded body, or the response itself when `raw`"
    try: resp.raise_for_status()
    except httpx2.HTTPStatusError as e:
        e.args = (f"{e}\n{resp.text}",)
        raise
    return resp if raw else self._decode(resp)

@patch
async def request(self:AsyncTransport, method, url, *, headers=None, params=None, json=None, data=None,
    files=None, content=None, raw=False):
    "Execute a request and decode JSON/text/binary response."
    resp = await self._attempt(method, url, headers=self._request_headers(headers, files=files),
        params=params, json=json, data=data, files=files, content=content)
    return self._result(resp, raw)

In [ ]:
t = AsyncTransport()
url = "https://api.openai.com/v1/chat/completions"
hdrs = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}
body = dict(model="gpt-4o-mini", max_tokens=5, messages=[dict(role="user", content="Say hi")])
resp = await t.request("POST", url, headers=hdrs, json=body)
resp

{'id': 'chatcmpl-Dz8eDU1VBkguT55jGrdyOSGBZzAMN',
 'object': 'chat.completion',
 'created': 1783463621,
 'model': 'gpt-4o-mini-2024-07-18',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Hi there! How can',
    'refusal': None,
    'annotations': []},
   'logprobs': None,
   'finish_reason': 'length'}],
 'usage': {'prompt_tokens': 9,
  'completion_tokens': 5,
  'total_tokens': 14,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 0,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': 'fp_a5a1892700'}

Redirects are followed by default (`follow_redirects=False` to disable) -- many APIs depend on this, e.g. GitHub 301s renamed repos and explicitly tells clients to follow:

In [ ]:
r = await AsyncTransport().request("GET", "https://jsonplaceholder.typicode.com/todos/1")
test_eq(r['id'], 1)

The fresh client per request is what keeps a transport loop-agnostic: a persistent client's pooled connections die with the event loop that created them, so a client you manage via `client=` must stay within a single loop.

In [ ]:
class Flaky:
    "A mock server whose first connection stalls"
    def __init__(self): self.calls = 0
    def __call__(self, req):
        self.calls += 1
        if self.calls == 1: raise httpx2.ConnectTimeout('stalled')
        return httpx2.Response(200, json={'ok': True})

The first attempt meets the stall and the second answers, so the caller sees only the result:


In [ ]:
flaky = Flaky()
ft = AsyncTransport(client=httpx2.AsyncClient(transport=httpx2.MockTransport(flaky)))
test_eq(await ft.request('GET', 'https://example.com/'), {'ok': True})
flaky.calls

With `retries=0` there is no second attempt, and the stall is the result:

In [ ]:
once = AsyncTransport(client=httpx2.AsyncClient(transport=httpx2.MockTransport(Flaky())), retries=0)
with expect_fail(httpx2.ConnectTimeout): await once.request('GET', 'https://example.com/')

### `SyncTransport`

A sync twin of `AsyncTransport` over `httpx2.Client`, for callers that can't (or don't want to) be async. It inherits header merging and response decoding, and keeps the fresh-client-per-request design. SSE streaming is deliberately not duplicated: on the sync transport `stream` raises, pointing at the bridge below.

In [ ]:
class SyncTransport(AsyncTransport):
    "Sync twin of `AsyncTransport`, over `httpx2.Client`. SSE needs the async transport, so `stream` raises."
    def stream(self, *args, **kwargs):
        raise TypeError("SSE streaming needs `AsyncTransport`; drive it from sync code with `fastcore.aio.iter_sync`")

    def _client(self):
        if self.client: return nullcontext(self.client)
        return httpx2.Client(timeout=self.timeout, follow_redirects=self.follow_redirects, verify=self.verify)

    def close(self):
        if self.client: self.client.close()

    def _send(self, method, url, **kw):
        with self._client() as client: return client.request(method, url, **kw)

    def _attempt(self, method, url, **kw):
        for i in range(self.retries):
            try: return self._send(method, url, **kw)
            except (httpx2.ConnectTimeout, httpx2.ConnectError): time.sleep(0.5 * 2**i)
        return self._send(method, url, **kw)

    def request(self, method, url, *, headers=None, params=None, json=None, data=None,
        files=None, content=None, raw=False):
        "Sync version of `AsyncTransport.request`."
        resp = self._attempt(method, url, headers=self._request_headers(headers, files=files),
            params=params, json=json, data=data, files=files, content=content)
        return self._result(resp, raw)

In [ ]:
st = SyncTransport()
r = st.request("GET", "https://jsonplaceholder.typicode.com/todos/1")
test_eq(r['id'], 1)
sflaky = SyncTransport(client=httpx2.Client(transport=httpx2.MockTransport(Flaky())))
test_eq(sflaky.request('GET', 'https://example.com/'), {'ok': True})
with expect_fail(Exception, 'iter_sync'): st.stream('GET', 'https://example.com')

For the pieces that stay async-only, `fastcore.aio` provides stdlib-only bridges: `run_sync` runs any awaitable from sync code (on a shared background event loop, so it works inside Jupyter too), and `iter_sync` does the same for async generators, here driving the async transport's `stream`:

In [ ]:
from fastcore.aio import iter_sync

In [ ]:
chunks = list(iter_sync(t.stream("POST", "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
    json=dict(model="gpt-4o-mini", stream=True, max_tokens=5, messages=[dict(role="user", content="Say hi")]))))
chunks[:2]

[{'id': 'chatcmpl-Dz8eGJPzFOS8nHwNYhlOCtISKVX0A',
  'object': 'chat.completion.chunk',
  'created': 1783463624,
  'model': 'gpt-4o-mini-2024-07-18',
  'service_tier': 'default',
  'system_fingerprint': 'fp_a5a1892700',
  'choices': [{'index': 0,
    'delta': {'role': 'assistant', 'content': '', 'refusal': None},
    'logprobs': None,
    'finish_reason': None}],
  'obfuscation': 'E6666m'},
 {'id': 'chatcmpl-Dz8eGJPzFOS8nHwNYhlOCtISKVX0A',
  'object': 'chat.completion.chunk',
  'created': 1783463624,
  'model': 'gpt-4o-mini-2024-07-18',
  'service_tier': 'default',
  'system_fingerprint': 'fp_a5a1892700',
  'choices': [{'index': 0,
    'delta': {'content': 'Hi'},
    'logprobs': None,
    'finish_reason': None}],
  'obfuscation': 'VZcpd2'}]

In [ ]:
chunks = [chunk async for chunk in t.stream("POST", "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
    json=dict(model="gpt-4o-mini", stream=True, max_tokens=5, messages=[dict(role="user", content="Say hi")]))]
chunks[:3]


[{'id': 'chatcmpl-Dz8eHm1tdv4bPryq1Q50ohrfJo9O2',
  'object': 'chat.completion.chunk',
  'created': 1783463625,
  'model': 'gpt-4o-mini-2024-07-18',
  'service_tier': 'default',
  'system_fingerprint': 'fp_a5a1892700',
  'choices': [{'index': 0,
    'delta': {'role': 'assistant', 'content': '', 'refusal': None},
    'logprobs': None,
    'finish_reason': None}],
  'obfuscation': 'fVk9G4'},
 {'id': 'chatcmpl-Dz8eHm1tdv4bPryq1Q50ohrfJo9O2',
  'object': 'chat.completion.chunk',
  'created': 1783463625,
  'model': 'gpt-4o-mini-2024-07-18',
  'service_tier': 'default',
  'system_fingerprint': 'fp_a5a1892700',
  'choices': [{'index': 0,
    'delta': {'content': 'Hi'},
    'logprobs': None,
    'finish_reason': None}],
  'obfuscation': '5ZZTdd'},
 {'id': 'chatcmpl-Dz8eHm1tdv4bPryq1Q50ohrfJo9O2',
  'object': 'chat.completion.chunk',
  'created': 1783463625,
  'model': 'gpt-4o-mini-2024-07-18',
  'service_tier': 'default',
  'system_fingerprint': 'fp_a5a1892700',
  'choices': [{'index': 0,
    

## Small REST clients

A little REST API wrapper needs a base URL, the HTTP verbs, and nothing else. `HttpCli` carries that shape over `SyncTransport`. The verbs are attributes that call `body`: kwargs become the payload, merged into the query params on `get` and `delete` and into the JSON body on `post`, `put`, and `patch`. `params=` and `json=` remain available for mixed calls, `useparams_=True` overrides the destination per call, and `path` is positional-only so even a payload key named `path` works. `AsyncHttpCli` is the same shape over `AsyncTransport`, and its verbs return awaitables.

In [ ]:
class HttpCli:
    "Small REST client over `SyncTransport`: verb attributes (`get`, `post`, `put`, `patch`, `delete`) call `body`, with kwargs as the payload"
    tcls = SyncTransport
    _verbs = ('get','post','put','patch','delete')
    @delegates(SyncTransport)
    def __init__(self, base_url, transport=None, **kwargs):
        self.base_url = base_url.rstrip('/')
        self.transport = transport or self.tcls(**kwargs)

    def _url(self, path): return f"{self.base_url}/{str(path).lstrip('/')}" if path else self.base_url
    def _req(self, method, path='', **kw): return self.transport.request(method, self._url(path), **kw)
    def body(self, method, path='', /, params=None, json=None, useparams_=False, **kw):
        "Send a request with `kw` merged into `params` if `useparams_`, else into `json`"
        if useparams_: params = {**(params or {}), **kw}
        else: json = {**(json or {}), **kw}
        return self._req(method, path, params=params, json=json or None)
    def __getattr__(self, name):
        if name not in self._verbs: raise AttributeError(name)
        return partial(self.body, name.upper(), useparams_=name in ('get','delete'))
    def __dir__(self): return [*super().__dir__(), *self._verbs]

class AsyncHttpCli(HttpCli):
    "Async twin of `HttpCli`, over `AsyncTransport`: the verbs return awaitables"
    tcls = AsyncTransport
    async def _req(self, method, path='', **kw): return await self.transport.request(method, self._url(path), **kw)

In [ ]:
cli = HttpCli('https://jsonplaceholder.typicode.com')
t = cli.get('todos/1')
test_eq(t['id'], 1)
r = cli.post('posts', title='hi')
test_eq(r['title'], 'hi')

acli = AsyncHttpCli('https://jsonplaceholder.typicode.com')
test_eq((await acli.get('todos/1'))['id'], 1)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()